# Agent RAG — Assistant sur l'Intelligence Artificielle

Ce notebook construit pas à pas un **agent RAG** (Retrieval-Augmented Generation).

### C'est quoi le RAG ?
Plutôt que de laisser le modèle répondre de mémoire (et risquer d'inventer des choses),
je lui fournis les passages les plus pertinents extraits de mes propres documents PDF.
Il répond alors **uniquement à partir de ce contexte** : les réponses sont plus fiables
et je sais exactement d'où elles viennent.

### Les 4 grandes étapes
1. Je charge mes PDFs et je les découpe en morceaux
2. Je convertis ces morceaux en vecteurs (embeddings) et je les stocke dans FAISS
3. Quand je pose une question, FAISS trouve les morceaux les plus proches
4. J'envoie ces morceaux + ma question au modèle Gemini qui formule la réponse


In [1]:
#charger les bibliothèques nécessaires et la clé d'API
from dotenv import load_dotenv

#charger la clé d'API à partir du fichier .env
load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import FastEmbedEmbeddings

print("Importation des bibliothèques et chargement de la clé d'API réussis.")


/tmp/ipykernel_7473/625444878.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


Importation des bibliothèques et chargement de la clé d'API réussis.


## Etape 2 — Chargement des documents PDF

Je charge tous les fichiers `.pdf` presents dans le dossier `documents/`.

> **Important :** `documents` contiendra le nombre total de **pages** chargees,
> pas le nombre de fichiers PDF. 3 PDFs de 10 pages = 30 documents.


In [2]:
#charger les documents PDF à partir du répertoire spécifié
#chaque page est un document séparé, ce qui permet une meilleure granularité pour les recherches ultérieures
#documents -> c'est le nombre de pages de l'ensemble des PDF chargés, pas le nombre de fichiers PDF

loader = DirectoryLoader('documents/', glob='*.pdf', loader_cls=PyPDFLoader)
documents = loader.load()

print(f"{len(documents)} documents PDF chargés avec succès.")

91 documents PDF chargés avec succès.


## Etape 3 — Decoupe en morceaux (chunking)

Meme si j'ai deja decoupe par page, certaines pages peuvent etre tres longues.
Je les redecoupe en morceaux de **30 000 caracteres** avec un chevauchement de **300 caracteres**.

**Pourquoi chevaucher ?**
Si une idee est a cheval sur deux morceaux, le chevauchement fait que les deux morceaux
contiennent cette idee — je ne perds pas d'information a la frontiere.

**Pourquoi 30 000 ?**
Avec des chunks trop petits, les morceaux sont trop fragmentes et le modele
n'a pas assez de contexte pour formuler une bonne reponse.


In [3]:
#decouper les documents en morceaux de texte plus petits pour une meilleure gestion et recherche
text_splitter = RecursiveCharacterTextSplitter(chunk_size=30000, chunk_overlap=300)
texts = text_splitter.split_documents(documents)

print(f"{len(texts)} morceaux de texte créés à partir des documents PDF.")

91 morceaux de texte créés à partir des documents PDF.


## Etape 4 — Creation de la base vectorielle FAISS

C'est l'etape cle du RAG. Je transforme chaque morceau en **vecteur numerique**
qui represente le sens du texte, puis je stocke tout dans FAISS.

**Comment ca marche la recherche vectorielle ?**
Deux textes qui parlent de la meme chose ont des vecteurs proches dans l'espace mathematique.
Quand je pose une question, FAISS cherche les morceaux dont le vecteur est le plus proche
du vecteur de ma question — meme si les mots exacts sont differents.

**Avantage de FastEmbedEmbeddings :** tourne en local, 100% gratuit, aucun quota.


In [7]:
# Embeddings LOCAUX : gratuits, sans quota, sans API
embeddings = FastEmbedEmbeddings()

# Plus besoin de lots ni de pauses ! On fait tout d'un coup, en local
vectorstore = FAISS.from_documents(texts, embeddings)

vectorstore.save_local("src/faiss_index")
print("Base FAISS créée et sauvegardée localement avec succès.")


Base FAISS créée et sauvegardée localement avec succès.


## Etape 5 — Initialisation du modele et de la fonction RAG

Je configure les deux elements qui vont repondre a mes questions :
- Le **retriever** : cherche dans FAISS les morceaux les plus pertinents
- Le **LLM** (Large Language Model) : lit ces morceaux et formule la reponse


In [5]:
#poser des questions à l'agent RAG


llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash-lite", temperature=0.2)

#le chercheur va trouver les 4 morceaux de texte les plus pertinents pour la question posée en sens (on peut ajuster ce 
# nombre avec le paramètre "k" en fonction de la question et de la taille des morceaux de texte)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})


#fonction pour repondre à une question en utilisant le modèle de langage et les morceaux de texte pertinents
def repondre_a_question(question):
    #récupérer les morceaux de texte les plus pertinents pour la question posée dans les documents PDF
    morceaux_pertinents = retriever.invoke(question)

    #mettre les morceaux de texte pertinents dans le contexte pour que le modèle 
    #de langage puisse les utiliser pour répondre à la question
    contexte = "" 
    for passage in morceaux_pertinents:
        contexte += passage.page_content + "\n\n"

    #poser la question au modèle de langage en lui fournissant le contexte des morceaux de texte pertinents
    message = f"""Réponds à la question en utilisant SEULEMENT le contexte ci-dessous.
    Si la réponse n'y est pas, dis simplement que je n'' ai pas les compétences pour répondre à cette question.
    
    CONTEXTE:
    {contexte}
    QUESTION: {question}
    RÉPONSE:"""
        #envoyer la question et le contexte au modèle de langage pour obtenir une réponse
    reponse = llm.invoke(message)
        
        #afficher la réponse obtenue du modèle de langage
    #print("REPONSE OBTENUE DU MODÈLE DE LANGAGE :")
    print(reponse.content)
    
    #afficher les sources utilisées pour répondre à la question (les morceaux de texte pertinents)
    #print("\nSOURCES UTILISÉES :")
    #for passage in morceaux_pertinents:
        #print("-", passage.metadata.get("source", "inconnue"))  # Affiche les 200 premiers caractères de chaque passage pertinent
    



## Etape 6 — Tests

Je pose quelques questions pour verifier que tout fonctionne.


In [6]:
#exemple de question à poser à l'agent RAG
repondre_a_question("c'est quoi intelligence artificielle ?")
repondre_a_question("Quelles sont des applications concrètes de l'intelligence artificielle ?")
repondre_a_question("quelle est les origines ou l'histoire de l'intelligence artificielle ?")

L'intelligence artificielle (IA) est l'ensemble des systèmes informatiques capables d'effectuer des tâches typiquement associées à l'intelligence, telles que l'apprentissage, le raisonnement, la résolution de problèmes, la perception ou la prise de décision. C'est également le champ de recherche visant à développer de tels systèmes. L'IA est un domaine de l'informatique qui s'appuie sur des fondements mathématiques et des concepts issus des sciences cognitives, et vise à résoudre des problèmes à forte complexité logique ou algorithmique. Dans le langage courant, l'IA inclut les dispositifs imitant, simulant ou remplaçant l'homme dans certaines mises en œuvre de ses fonctions cognitives.
Les applications de l'IA couvrent de nombreux domaines, notamment les moteurs de recherche, les systèmes de recommandation, l'aide au diagnostic médical, la compréhension du langage naturel, les voitures autonomes, les chatbots, les outils de génération d'images, les outils de prise de décision automati